In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import sys
import pandas as pd

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(r'H:\phd stuff\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

In [ ]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]

In [ ]:
a = 1
medium = td.Medium(permittivity=11.56)
run = True

In [ ]:
lambdas =  1/np.array([0.15,0.85])

In [ ]:
folder_path=rf"./Structures"
polarization = "TE"
project_name = fr"20260519 Transmission gyromorphs - SHU 100 x 15 Aaron {polarization}"
empty=False
runtime_ps = 20e-12
min_steps_per_lambda = 20
id0 = ""
add_ref = True
Lx, Ly = 100, 15


In [ ]:
def get_scale_factor(data,N):
    xmin, xmax = data['x'].min(), data['x'].max()
    ymin, ymax = data['y'].min(), data['y'].max()
    Lx = xmax - xmin
    Ly = ymax - ymin
    A = Lx * Ly
    density = N / A
    scale = np.sqrt(density)
    print(f'  N = {N}')
    print(f'  x range: [{xmin:.6f}, {xmax:.6f}]  Lx = {Lx:.6f}')
    print(f'  y range: [{ymin:.6f}, {ymax:.6f}]  Ly = {Ly:.6f}')
    print(f'  Area A = {A:.6f}')
    print(f'  Density N/A = {density:.6f}')
    print(f'  Scale factor to get N/A = 1: multiply coords by sqrt(N/A) = {scale:.6f}')
    print(f'  (resulting box size: Lx*scale = {Lx*scale:.6f}, Ly*scale = {Ly*scale:.6f})')
    return {"scale": scale, "Lx": Lx, "Ly": Ly, "xmin": xmin, "xmax": xmax, "ymin": ymin, "ymax": ymax}

In [ ]:
data = []
scales_data = []
for l,file in enumerate((os.listdir(folder_path))):
    print(Path(file).stem)
    df = pd.read_csv(os.path.join(folder_path, file))
    scales = get_scale_factor(df, N=len(df))
    df['x'] -=(scales["xmax"]+scales["xmin"])/2
    df['y'] -=(scales["ymax"]+scales["ymin"])/2
    print(f'  After centering:')
    print(df["x"].max(),df["x"].min())
    print(df["y"].max(),df["y"].min())
    data.append(df)
    scales_data.append(scales["scale"])


In [ ]:

for k,item in enumerate(data):
        run_name = f"{Path(os.listdir(folder_path)[k]).stem} Transmission {lambdas[0]:.3g} - {lambdas[1]:.3g}"
        print(run_name)
        
        structure_1 = AM.loadAndRunStructure(key = tidy3dAPI
                    ,direction="z", lambda_range=lambdas,
                    box_size= 50,runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                   scaling=1,shuoff_condtion=1e-20, verbose=True,
                   monitors=["flux"],
                   freqs=400, 
                   source="planewave", 
                   width=0.35, ref_only=True
                   )


        sim = structure_1.sim

        boundaries= td.BoundarySpec(
                y=td.Boundary(plus=td.Absorber(num_layers=200),minus=td.Absorber(num_layers=200)),
                x=td.Boundary.periodic(),
                z=td.Boundary.periodic(),
            )
        sim = sim.copy(update={'boundary_spec':boundaries})
        sim = sim.copy(update={'sources':[]})
        sim = sim.copy(update={'monitors':[]})


        sim = sim.copy(update={'size':[Lx,Ly+40,0]})

        source = td.PlaneWave(
                source_time = td.GaussianPulse(
                    freq0=structure_1.freq0,
                    fwidth=structure_1.freqw
                ),
                size= (td.inf,0,td.inf),
                center=(0,-(Ly)/2 - lambdas[0],0),
                direction='+',
                pol_angle=np.pi/2 if polarization == "TM" else 0,
                name='planewave',
                )

        sim = sim.copy(update={'sources':[source]})

      
        monitor_1 = td.FluxMonitor(
                center = (
                                0,-((Ly)/2+3),0
                                ),
                size = (
                   td.inf,0,td.inf
                    ),
                freqs = structure_1.monitor_freqs,
                name='flux2' )

        monitor_2 = td.FluxMonitor(
                center = (
                                0,(Ly)/2+3,0
                                ),
                size = (
                   td.inf,0,td.inf
                    ),
                freqs = structure_1.monitor_freqs,
                name='flux1' 
            )

        sim = sim.copy(update={'monitors':[monitor_1,monitor_2]})

        cyl_group = []
        for x, y in zip(data[k]['x'], data[k]['y']):
            x,y = x*scales_data[k],y*scales_data[k]
            if np.abs(y)<=(Ly/2) and np.abs(x)<=(Lx/2):
                cyl_group.append(td.Cylinder(center=[x, y, 0], radius=0.189, length=td.inf))

        cylinders = td.Structure(geometry=td.GeometryGroup(geometries=cyl_group), medium=medium)


        if empty:
            sim = sim.copy(update={'structures':[],"grid_spec": td.GridSpec.uniform(dl=structure_1.dl)})
        else:
            sim = sim.copy(update={'structures':[cylinders],"grid_spec": td.GridSpec.uniform(dl=structure_1.dl)})

        sim_name = run_name
        fig, ax = plt.subplots(1, tight_layout=True, figsize=(16, 8))
        sim.plot(z=0, ax=ax)
        plt.show()

        if run:
            if add_ref:
                 id0 =web.upload(sim.copy(update={'structures':[]}), folder_name=project_name,task_name=fr"{sim_name}_0", verbose=True)
                 web.start(task_id = id0)
                 web.monitor(task_id=id0,verbose=True)
                 add_ref = False

            id =web.upload(sim, folder_name=project_name,task_name=sim_name, verbose=True)
            web.start(task_id = id)
            web.monitor(task_id=id,verbose=True)
            ids = id0+ '\n' + id
            structure_folder =Path(os.listdir(folder_path)[k]).stem
            file_path = rf"H:\phd stuff\tidy3d\data/{project_name}/{structure_folder}/{sim_name}.txt"
            # Check if the folder exists
            if not os.path.exists( fr"H:\phd stuff\tidy3d\data/{project_name}/{structure_folder}"):
                os.makedirs(fr"H:\phd stuff\tidy3d\data/{project_name}/{structure_folder}")
                print(fr"Folder 'H:\phd stuff\tidy3d\data/{project_name}/{structure_folder}' created successfully.")

            # Open file in write mode
            with open(file_path, "w") as f:
                # Write the string to the file
                f.write(ids)
        else:
            sim.plot_3d()
            id =web.upload(sim,task_name=sim_name, verbose=True)
            print(web.estimate_cost(id))
            raise TypeError("Program ends here")

